In [62]:
import sys

from pathlib import Path

print(sys.executable)
print(Path.cwd())

c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
c:\dev\llm-data-analysis-course\notebooks


### STEP 1. 예측 문제와 예측 시점을 먼저 정의합니다

무엇을 예측하는가?

예측 단위는 무엇인가?

언제 예측하는가?

그 시점에 실제로 알 수 있는 정보는 무엇인가?

### STEP 2. Feature Leakage Audit을 수행합니다

STEP 2는 예측할 때 실제로 알 수 있는 정보만 모델에 넣기 위한 검사예요. 주문 금액을 예측하면서 그 계산 재료인 수량·단가를 넣으면 정답을 미리 보여주는 셈이에요. 이런 정보가 섞이면 점수는 높아 보여도 실제 예측에는 쓸 수 없어서, 항목마다 사용 가능 여부를 확인합니다.

In [63]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 공식 실습 저장소의 전처리 완료 데이터 경로
processed_dir = Path(r"C:\dev\llm-data-analysis-course-official\data\processed")

file_names = {
    "customers": "customers_clean.csv",
    "products": "products_clean.csv",
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
}

# 파일이 없으면 다른 데이터를 대신 사용하지 않고 오류로 알려준다.
missing_files = [
    name for name in file_names.values()
    if not (processed_dir / name).is_file()
]
if missing_files:
    raise FileNotFoundError(
        f"processed 폴더에 없는 파일: {missing_files}\n"
        "먼저 공식 저장소에서 python scripts/prepare_ch09_data.py를 실행해 주세요."
    )

# processed CSV를 직접 읽는다.
tables = {
    name: pd.read_csv(processed_dir / file_name)
    for name, file_name in file_names.items()
}
existing_columns = {
    column
    for df in tables.values()
    for column in df.columns
}

# 각 항목: 예측 시점 사용 가능, Target 계산 재료, 사후 정보/ID, 최종 사용, 판단 이유
rules = {
    "order_month": ("예", "아니오", "아니오", "예", "주문 날짜에서 월을 생성할 수 있다."),
    "order_dayofweek": ("예", "아니오", "아니오", "예", "주문 날짜에서 요일을 생성할 수 있다."),
    "age": ("예", "아니오", "아니오", "예", "예측 시점에 알 수 있는 고객 특성이다."),
    "payment_method": ("예", "아니오", "아니오", "예", "이번 과제에서는 예측 시점에 결제 수단을 안다고 가정한다."),
    "gender": ("예", "아니오", "아니오", "예", "예측 시점에 알 수 있는 고객 특성이다."),
    "city": ("예", "아니오", "아니오", "예", "예측 시점에 알 수 있는 고객 특성이다."),
    "order_total": ("아니오", "예", "아니오", "아니오", "예측하려는 정답(Target) 자체이다."),
    "line_total": ("아니오", "예", "아니오", "아니오", "주문 금액을 합산하는 재료이다."),
    "quantity": ("아니오", "예", "아니오", "아니오", "주문 상세 수량으로 금액 계산 재료이다."),
    "unit_price": ("아니오", "예", "아니오", "아니오", "주문 상세 단가로 금액 계산 재료이다."),
    "item_count": ("아니오", "예", "예", "아니오", "주문 상세가 확정된 뒤 집계하는 값이다."),
    "total_quantity": ("아니오", "예", "예", "아니오", "주문 상세 수량을 집계한 값이다."),
    "avg_unit_price": ("아니오", "예", "예", "아니오", "주문 상세 단가를 집계한 값이다."),
    "order_status": ("아니오", "아니오", "예", "아니오", "주문 이후 바뀔 수 있는 상태이다."),
    "order_id": ("예", "아니오", "예", "아니오", "주문 식별자이다."),
    "customer_id": ("예", "아니오", "예", "아니오", "고객 식별자이다."),
    "product_id": ("아니오", "아니오", "예", "아니오", "상품 식별자이다."),
    "order_date": ("예", "아니오", "아니오", "아니오", "시간 분할과 월·요일 생성에 사용한다."),
}

rows = []
# 과제의 항목과 processed 데이터에 실제로 있는 추가 컬럼을 모두 확인한다.
features = list(rules) + sorted(existing_columns - set(rules))

for feature in features:
    if feature in rules:
        available, material, later_or_id, selected, reason = rules[feature]

        if feature not in existing_columns:
            if feature in {"order_month", "order_dayofweek"} and "order_date" in existing_columns:
                selected = "예 (생성 필요)"
                reason += " processed에는 없으며 order_date에서 생성해야 한다."
            elif selected == "예":
                selected = "보류 (컬럼 없음)"
                reason += " processed에서 해당 컬럼을 찾지 못했다."
            else:
                reason += " processed에는 없으며 계산 또는 집계 단계에서 나타날 수 있다."
    else:
        # 안내문에 없는 컬럼은 뜻을 확인할 때까지 모델 입력에서 보류한다.
        available, material, later_or_id, selected = (
            "확인 필요", "확인 필요", "확인 필요", "보류"
        )
        reason = "processed에 존재한다. 예측 시점과 Target의 관계를 확인해야 한다."

    rows.append({
        "feature": feature,
        "예측 시점 사용 가능": available,
        "Target 계산 재료": material,
        "사후 정보/ID": later_or_id,
        "최종 사용": selected,
        "판단 이유": reason,
    })

feature_audit = pd.DataFrame(rows)

print("사용한 데이터 폴더:", processed_dir)
print("읽은 파일:", ", ".join(file_names.values()))
display(feature_audit)

사용한 데이터 폴더: C:\dev\llm-data-analysis-course-official\data\processed
읽은 파일: customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv


,feature,예측 시점 사용 가능,Target 계산 재료,사후 정보/ID,최종 사용,판단 이유
0,order_month,예,아니오,아니오,예,주문 날짜에서 월을 생성할 수 있다.
1,order_dayofweek,예,아니오,아니오,예,주문 날짜에서 요일을 생성할 수 있다.
2,age,예,아니오,아니오,예,예측 시점에 알 수 있는 고객 특성이다.
3,payment_method,예,아니오,아니오,예,이번 과제에서는 예측 시점에 결제 수단을 안다고 가정한다.
4,gender,예,아니오,아니오,예,예측 시점에 알 수 있는 고객 특성이다.
5,city,예,아니오,아니오,예,예측 시점에 알 수 있는 고객 특성이다.
6,order_total,아니오,예,아니오,아니오,예측하려는 정답(Target) 자체이다. processed에는 없으며 계산 또는 집...
7,line_total,아니오,예,아니오,아니오,주문 금액을 합산하는 재료이다.
8,quantity,아니오,예,아니오,아니오,주문 상세 수량으로 금액 계산 재료이다.
9,unit_price,아니오,예,아니오,아니오,주문 상세 단가로 금액 계산 재료이다.


In [64]:
# 위 셀에서 만든 feature_audit 표의 보류 항목을 검토 결과로 수정한다.
updates = {
    "category": ("아니오", "아니오", "예", "아니오",
                 "주문한 상품을 알아야 연결할 수 있는 상품 정보이다."),
    "name": ("예", "아니오", "예", "아니오",
             "고객 이름은 개인 식별 정보이므로 모델 입력에서 제외한다."),
    "order_item_id": ("아니오", "아니오", "예", "아니오",
                      "주문 상세 행의 식별자이다."),
    "price": ("아니오", "아니오", "예", "아니오",
              "주문 상품을 알아야 연결할 수 있다. Target 계산에는 unit_price를 사용한다."),
    "product_name": ("아니오", "아니오", "예", "아니오",
                     "주문한 상품을 알아야 연결할 수 있는 상품 정보이다."),
    "signup_date": ("예", "아니오", "아니오", "아니오",
                    "가입일은 미리 알 수 있지만 이번 모델의 선택 feature가 아니다."),
}

columns = [
    "예측 시점 사용 가능",
    "Target 계산 재료",
    "사후 정보/ID",
    "최종 사용",
    "판단 이유",
]

for feature, values in updates.items():
    feature_audit.loc[feature_audit["feature"] == feature, columns] = values

display(feature_audit)

,feature,예측 시점 사용 가능,Target 계산 재료,사후 정보/ID,최종 사용,판단 이유
0,order_month,예,아니오,아니오,예,주문 날짜에서 월을 생성할 수 있다.
1,order_dayofweek,예,아니오,아니오,예,주문 날짜에서 요일을 생성할 수 있다.
2,age,예,아니오,아니오,예,예측 시점에 알 수 있는 고객 특성이다.
3,payment_method,예,아니오,아니오,예,이번 과제에서는 예측 시점에 결제 수단을 안다고 가정한다.
4,gender,예,아니오,아니오,예,예측 시점에 알 수 있는 고객 특성이다.
5,city,예,아니오,아니오,예,예측 시점에 알 수 있는 고객 특성이다.
6,order_total,아니오,예,아니오,아니오,예측하려는 정답(Target) 자체이다. processed에는 없으며 계산 또는 집...
7,line_total,아니오,예,아니오,아니오,주문 금액을 합산하는 재료이다.
8,quantity,아니오,예,아니오,아니오,주문 상세 수량으로 금액 계산 재료이다.
9,unit_price,아니오,예,아니오,아니오,주문 상세 단가로 금액 계산 재료이다.


In [65]:
selected = feature_audit.loc[feature_audit["최종 사용"] == "예", "feature"]
forbidden = ["order_total","line_total","quantity","unit_price","item_count",
               "total_quantity","avg_unit_price","order_status","order_id",
               "customer_id","product_id"]
print(len(set(selected) & set(forbidden)))  # 0이면 OK

0


## 2. Feature Leakage Audit

### 결과 관찰

`data/processed`의 고객·상품·주문·주문 상세 데이터를 바탕으로 24개 항목을 수동 검토했다. 이 표에는 processed 데이터의 컬럼과 모델링 과정에서 생성하거나 검토할 항목이 함께 포함되어 있다.

`최종 사용 = 예`인 항목은 `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city`의 6개이며, 나머지 18개는 제외했다. 선택한 feature와 금지 feature의 교집합을 코드로 계산한 결과는 **0개**였다.

### 나의 해석과 판단

- **Target 자체:** `order_total`은 예측할 정답이므로 입력에서 제외한다. 주문 상세 금액을 합산해 생성하는 값이다.
- **계산 재료·대리변수:** `line_total`, `quantity`, `unit_price`, `item_count`, `total_quantity`, `avg_unit_price`는 주문 금액의 계산 재료이거나 주문 상세를 반영하므로 제외한다.
- **사후 정보:** `order_status`는 예측 시점 이후에 바뀔 수 있어 제외한다.
- **식별 정보:** `order_id`, `customer_id`, `product_id`, `order_item_id`는 식별자이며, `name`은 개인 식별 정보이므로 제외한다.

`category`, `price`, `product_name`은 이번 예측 시점에 주문할 상품을 알 수 없으므로 사용하지 않는다. 상품표의 `price`는 Target 계산에 사용하는 주문 상세의 `unit_price`와 구분한다. 원본 `order_date`는 모델에 직접 넣지 않고 시간 분할과 월·요일 생성에 사용한다. `signup_date`는 예측 전에 알 수 있지만 이번 기본 모델에는 포함하지 않았다.

가장 위험하다고 판단한 항목은 `line_total`이다. 이를 주문별로 합산하면 Target인 `order_total`이 되므로 모델 입력에 넣으면 정답의 재료가 섞인다.

### 업무·분석적 의미

정답의 재료나 주문 이후의 정보를 입력하면 평가 점수가 실제 예측 능력보다 높게 보일 수 있다. 금지 feature와 겹치지 않는 6개 입력으로 평가해야 실제 사용 상황에 가까운 결과를 판단할 수 있다.

### 한계와 추가 확인 사항

`payment_method`는 이번 과제에서 예측 시점에 이미 알 수 있다고 가정한다. 실제 서비스에서는 결제 수단이 언제 확정되는지 확인해야 한다. 상품 정보 역시 주문 전에 확정되는 업무 흐름인지에 따라 사용 가능 여부를 다시 검토할 수 있다.

이 표는 24개 항목을 검토한 수동 표이다. 공식 자동 Evidence CSV의 실제 내용과 행 수는 해당 파일을 생성하고 열어본 뒤 별도로 확인한다.

### STEP 3. Target 생성과 관계 검증을 확인합니다

Target은 주문 상세 데이터에서 만듭니다.

In [66]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 1. 공식 저장소의 전처리 완료 데이터를 읽는다.
data_dir = Path(r"C:\dev\llm-data-analysis-course-official\data\processed")

orders = pd.read_csv(data_dir / "orders_clean.csv")
order_items = pd.read_csv(data_dir / "order_items_clean.csv")
customers = pd.read_csv(data_dir / "customers_clean.csv")

# 2. 주문 상세의 수량, 단가, 저장된 금액을 숫자로 변환한다.
items = order_items.copy()
for column in ["quantity", "unit_price", "line_total"]:
    items[column] = pd.to_numeric(items[column], errors="coerce")

# line_total = quantity × unit_price를 직접 계산해 저장된 값과 비교한다.
items["calculated_line_total"] = items["quantity"] * items["unit_price"]

invalid_amount_rows = int(
    items[["quantity", "unit_price", "line_total"]].isna().any(axis=1).sum()
)
line_total_mismatch = int(
    ((items["line_total"] - items["calculated_line_total"]).abs() > 1e-6).sum()
)

# 3. 같은 주문의 상세 금액을 합쳐 Target을 만든다.
# 이 값은 정답 검증에만 사용하고 모델의 입력 feature에는 넣지 않는다.
order_totals = (
    items.groupby("order_id", as_index=False)["calculated_line_total"]
    .sum()
    .rename(columns={"calculated_line_total": "order_total"})
)

# 4. 주문과 Target이 주문 ID 기준으로 1:1 연결되는지 확인한다.
order_ids = set(orders["order_id"].dropna())
target_ids = set(order_totals["order_id"].dropna())

duplicate_order_ids = int(orders["order_id"].duplicated().sum())
missing_order_ids = int(orders["order_id"].isna().sum())
missing_item_order_ids = int(items["order_id"].isna().sum())
orders_without_target = len(order_ids - target_ids)
targets_without_order = len(target_ids - order_ids)

# 5. 각 주문의 고객 ID가 고객표의 한 행에 연결되는지 확인한다.
duplicate_customer_ids = int(customers["customer_id"].duplicated().sum())
missing_customer_ids = int(customers["customer_id"].isna().sum())
orders_without_customer = int(
    (~orders["customer_id"].isin(customers["customer_id"])).sum()
)

# 6. 검증 결과를 한눈에 볼 수 있도록 정리한다.
checks = [
    ("line_total 계산 일치", invalid_amount_rows + line_total_mismatch,
     f"숫자 결측/변환 실패 {invalid_amount_rows}건, 계산 불일치 {line_total_mismatch}건"),
    ("orders ↔ target: 1대1", duplicate_order_ids + missing_order_ids
     + missing_item_order_ids + orders_without_target + targets_without_order,
     f"중복 주문 ID {duplicate_order_ids}건, Target 없는 주문 {orders_without_target}건, "
     f"부모 주문 없는 Target {targets_without_order}건"),
    ("orders → customers: 다대1", duplicate_customer_ids
     + missing_customer_ids + orders_without_customer,
     f"중복 고객 ID {duplicate_customer_ids}건, 연결되지 않는 주문 {orders_without_customer}건"),
]

validation = pd.DataFrame(
    [
        {"검증 항목": name, "문제 건수": count,
         "결과": "PASS" if count == 0 else "FAIL", "상세": detail}
        for name, count, detail in checks
    ]
)

print("계산식: line_total = quantity × unit_price")
print("계산식: order_total = 같은 order_id의 line_total 합계")
print(f"주문 {len(orders)}건 → 생성된 Target {len(order_totals)}건")
display(validation)

print("생성한 Target 예시:")
display(order_totals.head())

계산식: line_total = quantity × unit_price
계산식: order_total = 같은 order_id의 line_total 합계
주문 300건 → 생성된 Target 300건


,검증 항목,문제 건수,결과,상세
0,line_total 계산 일치,0,PASS,"숫자 결측/변환 실패 0건, 계산 불일치 0건"
1,orders ↔ target: 1대1,0,PASS,"중복 주문 ID 0건, Target 없는 주문 0건, 부모 주문 없는 Target 0건"
2,orders → customers: 다대1,0,PASS,"중복 고객 ID 0건, 연결되지 않는 주문 0건"


생성한 Target 예시:


,order_id,order_total
0,1,1436000
1,2,756000
2,3,1435000
3,4,498000
4,5,941000


## 3. Target 생성과 관계 검증

### 계산 기준

- `line_total = quantity × unit_price`
- `order_total = 같은 order_id의 line_total 합계`
- 예측 단위: 주문 1건
- 생성된 Target: 300건 (주문 300건)

### 검증 결과

- 저장된 `line_total`과 재계산한 값의 불일치: **0건 (PASS)**
- `orders ↔ order_total` 1대1 관계: **Target 누락·부모 주문 누락·중복 주문 ID 모두 0건 (PASS)**
- `orders → customers` 다대1 관계: **중복 고객 ID·연결되지 않는 주문 모두 0건 (PASS)**

### 나의 해석과 판단

처음에는 기존 processed 파일에서 연결되지 않는 주문과 주문 상세가 발견됐다. 공식 입력 준비 스크립트로 원본 데이터를 다시 검증하고 processed 파일을 생성한 뒤, 세 검증 항목이 모두 PASS로 바뀌었다.

관계를 확인하지 않고 병합하면 일부 주문이 사라지거나 중복되어 Target 건수와 금액이 잘못될 수 있다. `order_total`과 그 계산 재료는 정답 생성에만 사용하고 모델의 입력 feature에서는 제외한다.

### STEP 4. Train과 Final Test를 시간 순서로 분리합니다

이번 장에서는 무작위 분할보다 시간 순서를 사용합니다.

### 목적
과거 주문으로 모델을 학습하고, 이후 주문으로 예측 성능을 평가한다.

### 진행 방법
`order_date`를 기준으로 주문을 오래된 순서로 정렬한다. 앞 기간은 Train, 마지막 기간은 Final Test로 나눈다. 같은 날짜의 주문은 한 그룹에만 배정한다.

### 확인할 것
- Train과 Final Test의 시작일·종료일 및 주문 건수
- Train의 마지막 날짜가 Final Test의 첫 날짜보다 이전인지
- 양쪽에 겹치는 주문 날짜가 없는지

### STEP 4. Train과 Final Test를 시간 순서로 나누는 이유

과거 주문 데이터로 모델을 학습하고, 이후 주문 데이터로 예측 성능을 평가한다. 시험 문제의 정답을 미리 보고 공부하면 실력을 정확히 평가할 수 없는 것과 같다. 따라서 오래된 주문은 **Train(연습 문제)**, 최근 주문은 **Final Test(시험 문제)**로 나눈다. 같은 날짜의 주문은 두 그룹에 섞이지 않도록 한다. 이를 통해 새로운 주문을 예측할 때의 성능에 가깝게 평가할 수 있다.

In [67]:
import pandas as pd
from IPython.display import display

# 1. 주문 1건당 1행인 모델링 데이터를 만든다.
model_data = orders.merge(
    order_totals,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

# 고객 특성을 연결한다. 여러 주문이 같은 고객에 연결될 수 있다.
model_data = model_data.merge(
    customers[["customer_id", "age", "gender", "city"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 2. 주문 날짜를 날짜형으로 바꾸고, 오래된 순서로 정렬한다.
model_data["order_date"] = pd.to_datetime(model_data["order_date"])
model_data = model_data.sort_values(
    ["order_date", "order_id"]
).reset_index(drop=True)

# 모델 입력에 쓸 월·요일을 주문 날짜에서 만든다.
model_data["order_month"] = model_data["order_date"].dt.month
model_data["order_dayofweek"] = model_data["order_date"].dt.dayofweek

# 3. 서로 다른 주문 날짜를 정렬해 약 80% 지점의 날짜를 찾는다.
# 같은 날짜에 발생한 주문은 반드시 같은 그룹으로 보낸다.
model_data["split_day"] = model_data["order_date"].dt.normalize()
unique_days = model_data["split_day"].drop_duplicates().reset_index(drop=True)

split_index = int(len(unique_days) * 0.8)
split_index = max(1, min(split_index, len(unique_days) - 1))
test_start_day = unique_days.iloc[split_index]

# 기준 날짜보다 이르면 Train, 기준 날짜부터는 Final Test이다.
train_data = model_data.loc[
    model_data["split_day"] < test_start_day
].drop(columns="split_day").copy()

final_test_data = model_data.loc[
    model_data["split_day"] >= test_start_day
].drop(columns="split_day").copy()

# 4. 분할 결과와 날짜 중복 여부를 검증한다.
train_days = set(train_data["order_date"].dt.normalize())
test_days = set(final_test_data["order_date"].dt.normalize())
overlap_days = train_days & test_days

summary = pd.DataFrame([
    {
        "구분": "Train",
        "시작일": train_data["order_date"].min().date(),
        "종료일": train_data["order_date"].max().date(),
        "행 수": len(train_data),
    },
    {
        "구분": "Final Test",
        "시작일": final_test_data["order_date"].min().date(),
        "종료일": final_test_data["order_date"].max().date(),
        "행 수": len(final_test_data),
    },
])

display(summary)
print("전체 주문 수:", len(model_data))
print("같은 날짜가 양쪽에 겹친 개수:", len(overlap_days))
print(
    "Train 마지막 날짜 < Final Test 첫 날짜:",
    train_data["order_date"].max().normalize()
    < final_test_data["order_date"].min().normalize()
)

# 조건이 어긋나면 다음 단계로 넘어가지 않도록 한다.
assert len(train_data) + len(final_test_data) == len(model_data)
assert len(overlap_days) == 0
assert train_data["order_date"].max().normalize() < final_test_data["order_date"].min().normalize()

,구분,시작일,종료일,행 수
0,Train,2025-07-09,2026-04-13,244
1,Final Test,2026-04-14,2026-07-08,56


전체 주문 수: 300
같은 날짜가 양쪽에 겹친 개수: 0
Train 마지막 날짜 < Final Test 첫 날짜: True


## STEP 4. Train / Final Test 시간 순서 분할 결과

### 결과 관찰

| 구분 | 시작일 | 종료일 | 주문 수 |
| --- | --- | --- | ---: |
| Train | 2025-07-09 | 2026-04-13 | 244건 |
| Final Test | 2026-04-14 | 2026-07-08 | 56건 |

- 전체 주문: **300건**
- 양쪽에 겹치는 주문 날짜: **0개**
- Train 마지막 날짜 < Final Test 첫 날짜: **True**

### 나의 해석과 판단

Train의 마지막 날짜가 Final Test의 첫 날짜보다 이르고, 같은 날짜가 양쪽에 겹치지 않는다. 따라서 과거 주문으로 학습하고 이후 주문으로 평가하는 시간 순서가 지켜졌다.

### 업무·분석적 의미

이 분할은 과거 데이터로 새로운 주문을 예측하는 실제 사용 상황에 가깝다. Final Test는 모델 선택에 사용하지 않고 마지막 성능 평가에만 사용한다.

In [68]:
# STEP 5-1. 모델에 넣을 데이터 확인하기

import pandas as pd
from IPython.display import display

# 숫자형과 범주형 feature를 구분한다.
numeric_features = ["order_month", "order_dayofweek", "age"]
categorical_features = ["payment_method", "gender", "city"]
feature_columns = numeric_features + categorical_features

# Train 데이터에서 입력(X)과 정답(y)을 분리한다.
X_train = train_data[feature_columns].copy()
y_train = train_data["order_total"].copy()

# 각 입력 컬럼의 자료형과 비어 있는 값의 개수를 확인한다.
feature_check = pd.DataFrame({
    "feature": X_train.columns,
    "구분": [
        "숫자형" if column in numeric_features else "범주형"
        for column in X_train.columns
    ],
    "자료형": [str(X_train[column].dtype) for column in X_train.columns],
    "Train 결측치": [int(X_train[column].isna().sum()) for column in X_train.columns],
})

print("Train 입력 크기:", X_train.shape)
print("Train 정답 수:", len(y_train))
display(feature_check)

Train 입력 크기: (244, 6)
Train 정답 수: 244


,feature,구분,자료형,Train 결측치
0,order_month,숫자형,int32,0
1,order_dayofweek,숫자형,int32,0
2,age,숫자형,int64,0
3,payment_method,범주형,str,0
4,gender,범주형,str,0
5,city,범주형,str,0


In [69]:
# STEP 5-2. 숫자형 전처리 Pipeline 만들기

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 숫자형 컬럼에 적용할 전처리 순서를 정의한다.
numeric_pipeline = Pipeline([
    # 비어 있는 값이 있으면 Train의 중앙값으로 채운다.
    ("imputer", SimpleImputer(strategy="median")),

    # 숫자형 컬럼의 크기를 표준화한다.
    ("scaler", StandardScaler()),
])

# 정의한 처리 순서를 확인한다.
for name, step in numeric_pipeline.steps:
    print(f"{name}: {step}")

imputer: SimpleImputer(strategy='median')
scaler: StandardScaler()


In [70]:
# STEP 5-3. 범주형 전처리 Pipeline 만들기

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 범주형 컬럼에 적용할 전처리 순서를 정의한다.
categorical_pipeline = Pipeline([
    # 비어 있는 값이 있으면 Train에서 가장 자주 나온 값으로 채운다.
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # 글자 범주를 모델이 사용할 수 있는 0과 1의 열로 바꾼다.
    # 이후 처음 보는 범주가 나와도 오류가 나지 않도록 한다.
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# 정의한 처리 순서를 확인한다. 아직 fit은 실행하지 않는다.
for name, step in categorical_pipeline.steps:
    print(f"{name}: {step}")

imputer: SimpleImputer(strategy='most_frequent')
encoder: OneHotEncoder(handle_unknown='ignore', sparse_output=False)


In [71]:
# STEP 5-4. 두 전처리를 하나로 묶기

from sklearn.compose import ColumnTransformer

# 컬럼 종류에 따라 서로 다른 전처리를 적용하도록 묶는다.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# 어떤 컬럼에 어떤 Pipeline이 연결됐는지 확인한다.
for name, pipeline, columns in preprocessor.transformers:
    print(f"{name}: {columns}")

numeric: ['order_month', 'order_dayofweek', 'age']
categorical: ['payment_method', 'gender', 'city']


In [72]:
# STEP 5-5. 전처리 설정 최종 확인

# 입력 6개가 전처리 설정에 모두 포함됐는지 확인한다.
configured_features = numeric_features + categorical_features

print("Train 입력 컬럼과 전처리 대상이 일치:",
      list(X_train.columns) == configured_features)

print("숫자형 컬럼 수:", len(numeric_features))
print("범주형 컬럼 수:", len(categorical_features))

# transformers_는 fit 이후에 만들어진다.
print("전처리 fit을 아직 실행하지 않음:",
      not hasattr(preprocessor, "transformers_"))

Train 입력 컬럼과 전처리 대상이 일치: True
숫자형 컬럼 수: 3
범주형 컬럼 수: 3
전처리 fit을 아직 실행하지 않음: True


## STEP 5. Pipeline 내부 전처리 준비

Train 데이터 244건에서 입력 feature 6개와 정답 `order_total`을 분리했다. 숫자형 3개(`order_month`, `order_dayofweek`, `age`)와 범주형 3개(`payment_method`, `gender`, `city`)의 결측치는 모두 0건이었다.

숫자형은 **중앙값 대체 → 표준화**, 범주형은 **최빈값 대체 → 원핫 인코딩** 순서의 Pipeline을 정의했다. 6개 입력 컬럼이 전처리 대상과 일치하는 것을 확인했다.

현재 전처리의 `fit`은 실행하지 않았다. 이후 모델과 묶어 교차검증할 때 각 Train 구간에서만 전처리를 학습시켜 데이터 누수를 막는다.

### STEP 6. Baseline과 후보 모델을 준비합니다

In [73]:
# STEP 6-1. 평균 예측 Baseline 만들기

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

# 전처리와 평균 예측 모델을 하나로 묶는다.
baseline_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DummyRegressor(strategy="mean")),
])

# Pipeline의 단계가 올바르게 연결됐는지 확인한다.
for name, step in baseline_model.steps:
    print(f"{name}: {type(step).__name__}")

preprocessor: ColumnTransformer
model: DummyRegressor


In [74]:
# STEP 6-2. Linear Regression 후보 만들기

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 전처리와 선형 회귀 모델을 하나로 묶는다.
linear_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LinearRegression()),
])

# Pipeline의 단계가 올바르게 연결됐는지 확인한다.
for name, step in linear_model.steps:
    print(f"{name}: {type(step).__name__}")

preprocessor: ColumnTransformer
model: LinearRegression


In [75]:
# STEP 6-3. Random Forest 후보 만들기

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# 전처리와 랜덤 포레스트 회귀 모델을 하나로 묶는다.
forest_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestRegressor(
        n_estimators=300,       # 결정나무 300개
        min_samples_leaf=5,     # 마지막 가지에 최소 5개 샘플
        random_state=42,        # 재실행해도 같은 결과가 나오도록 고정
        n_jobs=-1,              # 사용 가능한 CPU 코어 활용
    )),
])

# Pipeline의 단계가 올바르게 연결됐는지 확인한다.
for name, step in forest_model.steps:
    print(f"{name}: {type(step).__name__}")

preprocessor: ColumnTransformer
model: RandomForestRegressor


In [76]:
# STEP 6-4. 세 모델을 한곳에 모아 확인하기

# 준비한 모델 3개를 이름과 함께 모은다.
models = {
    "Baseline Mean": baseline_model,
    "Linear Regression": linear_model,
    "Random Forest": forest_model,
}

# 각 모델의 전처리와 회귀 모델이 연결됐는지 확인한다.
for name, pipeline in models.items():
    model_type = type(pipeline.named_steps["model"]).__name__
    fitted = hasattr(pipeline.named_steps["preprocessor"], "transformers_")
    print(f"{name}: {model_type}, 전처리 학습됨: {fitted}")

print("후보 모델 수:", len(models))

Baseline Mean: DummyRegressor, 전처리 학습됨: False
Linear Regression: LinearRegression, 전처리 학습됨: False
Random Forest: RandomForestRegressor, 전처리 학습됨: False
후보 모델 수: 3


## STEP 6. Baseline과 후보 모델 준비

전처리 Pipeline을 연결한 회귀 모델 3개를 준비했다.

| 모델 | 역할 |
| --- | --- |
| Baseline Mean (`DummyRegressor`) | Train의 평균 주문 금액을 예측하는 비교 기준 |
| Linear Regression | 첫 번째 회귀 후보 |
| Random Forest | 두 번째 회귀 후보 |

세 모델 모두 전처리 학습 상태가 `False`인 것을 확인했다. 다음 단계에서는 **Train 기간의 교차검증 결과**로 후보를 비교한다. Random Forest가 더 복잡하다는 이유만으로 성능이 좋다고 판단하지 않고, Baseline과 검증 지표를 비교한다.

### STEP 7. Train 기간에서 TimeSeriesSplit으로 후보를 비교합니다

In [77]:
# STEP 7-1. TimeSeriesSplit의 구간 확인

import pandas as pd
from IPython.display import display
from sklearn.model_selection import TimeSeriesSplit

# Train 데이터를 시간 순서에 따라 5번 나눌 준비를 한다.
time_cv = TimeSeriesSplit(n_splits=5)
fold_rows = []

for fold_number, (train_idx, validation_idx) in enumerate(
    time_cv.split(X_train), start=1
):
    # 각 회차에 해당하는 주문 날짜를 확인한다.
    train_dates = train_data.iloc[train_idx]["order_date"]
    validation_dates = train_data.iloc[validation_idx]["order_date"]

    fold_rows.append({
        "회차": fold_number,
        "학습 건수": len(train_idx),
        "학습 종료일": train_dates.max().date(),
        "검증 건수": len(validation_idx),
        "검증 시작일": validation_dates.min().date(),
        "검증 종료일": validation_dates.max().date(),
    })

display(pd.DataFrame(fold_rows))

,회차,학습 건수,학습 종료일,검증 건수,검증 시작일,검증 종료일
0,1,44,2025-09-05,40,2025-09-06,2025-10-20
1,2,84,2025-10-20,40,2025-10-21,2025-11-28
2,3,124,2025-11-28,40,2025-12-02,2026-01-13
3,4,164,2026-01-13,40,2026-01-14,2026-02-24
4,5,204,2026-02-24,40,2026-02-24,2026-04-13


In [78]:
# STEP 7-2. Train에서 세 모델 교차검증하기

import pandas as pd
from IPython.display import display
from sklearn.model_selection import cross_validate

cv_rows = []

for model_name, model in models.items():
    # 각 회차의 학습 구간에서 전처리와 모델을 함께 fit한다.
    result = cross_validate(
        model,
        X_train,
        y_train,
        cv=time_cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        error_score="raise",
    )

    # sklearn의 MAE 점수는 음수이므로 부호를 바꿔 실제 오차로 표시한다.
    mae_each_fold = -result["test_mae"]

    cv_rows.append({
        "모델": model_name,
        "CV MAE 평균": mae_each_fold.mean(),
        "CV MAE 표준편차": mae_each_fold.std(),
        "CV R² 평균": result["test_r2"].mean(),
    })

cv_summary = pd.DataFrame(cv_rows).sort_values("CV MAE 평균")
display(cv_summary)

,모델,CV MAE 평균,CV MAE 표준편차,CV R² 평균
0,Baseline Mean,423004.808743,44153.043466,-0.022821
2,Random Forest,437352.803929,26318.793429,-0.109687
1,Linear Regression,461386.286905,77792.799324,-0.310517


## STEP 7. Train 기간 교차검증 결과

### 결과 관찰

Train 데이터에서 5회 TimeSeriesSplit으로 세 모델을 비교했다.

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 |
| --- | ---: | ---: | ---: |
| Baseline Mean | 423,004.81 | 44,153.04 | -0.0228 |
| Random Forest | 437,352.80 | 26,318.79 | -0.1097 |
| Linear Regression | 461,386.29 | 77,792.80 | -0.3105 |

### 나의 해석과 판단

평균 오차는 Baseline Mean이 가장 낮았다. 따라서 현재 교차검증 결과만으로는 두 후보 모델이 평균 예측보다 개선됐다고 말할 수 없다.

비베이스라인 후보끼리 비교하면 Random Forest의 MAE 평균이 Linear Regression보다 낮고, 회차 간 MAE 표준편차도 더 작다. 세 모델의 CV R² 평균은 모두 음수였다. 다음 단계에서는 과제의 선택 기준에 따라 **비베이스라인 후보 중 하나를 Final Test 확인 전에 고정**해야 한다.

### 검증상의 한계

주문 행 기준 TimeSeriesSplit의 5회차에서는 학습과 검증에 `2026-02-24` 주문이 함께 포함됐다. 날짜 단위까지 엄격히 분리한 교차검증은 아니므로 이 한계를 기록한다. Train과 Final Test 사이에는 겹치는 날짜가 없다.

### STEP 8. Selected Model을 Final Test 전에 고정합니다

In [79]:
from sklearn.base import clone
from IPython.display import display

# Baseline은 비교 기준이므로 선택 대상에서 제외한다.
candidate_results = cv_summary.loc[
    cv_summary["모델"] != "Baseline Mean",
    ["모델", "CV MAE 평균", "CV MAE 표준편차", "CV R² 평균"]
].sort_values(["CV MAE 평균", "모델"]).reset_index(drop=True)

# Train CV의 평균 MAE가 가장 낮은 후보를 선택한다.
selected_model_name = candidate_results.loc[0, "모델"]
selected_cv_mae = candidate_results.loc[0, "CV MAE 평균"]

# 선택 모델을 학습 전 상태로 복사해 고정한다.
frozen_model = clone(models[selected_model_name])

display(candidate_results)
print("Selected Model:", selected_model_name)
print(f"선택 근거 CV MAE 평균: {selected_cv_mae:,.2f}")
print("선택 기준: 비베이스라인 후보 중 Train CV MAE 평균 최소")
print("Final Test 사용 여부: 사용하지 않음")
print(
    "고정 모델의 전처리 학습 전 상태:",
    not hasattr(frozen_model.named_steps["preprocessor"], "transformers_")
)

,모델,CV MAE 평균,CV MAE 표준편차,CV R² 평균
0,Random Forest,437352.803929,26318.793429,-0.109687
1,Linear Regression,461386.286905,77792.799324,-0.310517


Selected Model: Random Forest
선택 근거 CV MAE 평균: 437,352.80
선택 기준: 비베이스라인 후보 중 Train CV MAE 평균 최소
Final Test 사용 여부: 사용하지 않음
고정 모델의 전처리 학습 전 상태: True


## STEP 8. Train CV로 선택한 모델

- **Selected Model:** Random Forest
- **선택 기준:** 비베이스라인 후보 중 Train CV MAE 평균 최소
- **선택 모델의 CV MAE 평균:** 437,352.80
- **Linear Regression의 CV MAE 평균:** 461,386.29
- **Final Test를 보기 전에 선택했는가:** 예

Baseline Mean의 CV MAE 평균은 423,004.81로 세 모델 중 가장 낮았다. Baseline은 성능 비교 기준이므로, 비베이스라인 후보 중에서는 Random Forest를 선택했다. 이후 Final Test 결과를 보고 선택 모델을 바꾸지 않는다.

### STEP 9. Final Test에서는 Baseline과 Frozen Model만 평가합니다

In [80]:
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Final Test의 입력과 정답을 처음으로 분리한다.
X_final_test = final_test_data[feature_columns].copy()
y_final_test = final_test_data["order_total"].copy()

# 비교 대상은 Baseline과 이미 선택해 둔 모델 두 개뿐이다.
final_models = {
    "Baseline Mean": clone(baseline_model),
    selected_model_name: clone(frozen_model),
}

result_rows = []
final_predictions = {}

for model_name, model in final_models.items():
    # 전처리와 모델을 Train 데이터로만 학습한다.
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_final_test)
    final_predictions[model_name] = test_pred

    result_rows.append({
        "모델": model_name,
        "selection role": (
            "baseline" if model_name == "Baseline Mean"
            else "selected_by_train_cv"
        ),
        "Train MAE": mean_absolute_error(y_train, train_pred),
        "Test MAE": mean_absolute_error(y_final_test, test_pred),
        "Test RMSE": mean_squared_error(y_final_test, test_pred) ** 0.5,
        "Test R²": r2_score(y_final_test, test_pred),
    })

final_comparison = pd.DataFrame(result_rows)

# Baseline 대비 Final Test MAE가 얼마나 줄었는지 계산한다.
baseline_test_mae = final_comparison.loc[
    final_comparison["모델"] == "Baseline Mean", "Test MAE"
].iloc[0]

final_comparison["Baseline 대비 MAE 개선율(%)"] = (
    (baseline_test_mae - final_comparison["Test MAE"])
    / baseline_test_mae * 100
)

# Final Test에는 두 모델만 평가됐는지 확인한다.
assert set(final_comparison["모델"]) == {
    "Baseline Mean", selected_model_name
}
assert len(final_comparison) == 2

print("Final Test 주문 수:", len(X_final_test))
print("STEP 8에서 고정한 모델:", selected_model_name)
display(final_comparison.round(2))

Final Test 주문 수: 56
STEP 8에서 고정한 모델: Random Forest


,모델,selection role,Train MAE,Test MAE,Test RMSE,Test R²,Baseline 대비 MAE 개선율(%)
0,Baseline Mean,baseline,416266.19,443449.65,522189.70,-0.0,0.00
1,Random Forest,selected_by_train_cv,326438.48,458819.92,548190.45,-0.1,-3.47


In [81]:
print(
    final_comparison[["모델", "Test R²"]]
    .to_string(index=False, float_format=lambda value: f"{value:.4f}")
)

           모델  Test R²
Baseline Mean  -0.0011
Random Forest  -0.1033


## STEP 9. Final Test: Baseline vs Frozen Model

STEP 8에서 Train CV로 선택한 Random Forest를 고정한 뒤, Final Test 주문 56건에서 Baseline Mean과 비교했다. Final Test 결과를 보고 모델을 다시 선택하지 않았다.

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | 416,266.19 | 443,449.65 | 522,189.70 | -0.0011 | 0.00% |
| Random Forest | selected_by_train_cv | 326,438.48 | 458,819.92 | 548,190.45 | -0.1033 | -3.47% |

### 결과 관찰

Random Forest의 Train MAE는 Baseline보다 낮았지만, Final Test MAE는 **15,370.27원 더 높았다**. Test RMSE도 Random Forest가 더 높고, 두 모델의 Test R²는 모두 음수였다.

### 나의 해석과 판단

Random Forest는 이번 Final Test에서 Baseline보다 개선되지 않았다. `-3.47%`는 오차가 3.47% 줄었다는 뜻이 아니라 **Baseline보다 커졌다는 뜻**이다. Train에서 오차가 낮았던 것만으로 이후 주문도 더 잘 예측한다고 판단할 수 없다.

선택 모델은 Final Test 전에 고정했으므로 이 결과를 이유로 다른 후보로 바꾸지 않는다. 오차의 크기와 R²가 갖는 의미는 다음 단계에서 더 자세히 해석한다.

### STEP 10. MAE, RMSE, R²를 함께 해석합니다

### 결과 관찰

Final Test 56건에서 Baseline Mean의 MAE는 **443,449.65원**, RMSE는 **522,189.70원**, R²는 **-0.0011**이었다.

고정한 Random Forest의 MAE는 **458,819.92원**, RMSE는 **548,190.45원**, R²는 **-0.1033**이었다. Baseline 대비 MAE 개선율은 **-3.47%**로, 평균 오차가 15,370.27원 더 컸다.

### 나의 해석과 판단

- **MAE:** Random Forest는 주문 1건당 평균적으로 Baseline보다 더 크게 빗나갔다.
- **RMSE:** 두 모델 모두 RMSE가 MAE보다 크다. 일부 큰 오차가 있을 가능성이 있으므로 다음 단계에서 개별 주문을 확인해야 한다.
- **R²:** 두 값 모두 음수이며, Random Forest의 값이 더 낮다. 음수 R² 자체가 코드 오류를 뜻하지는 않는다. 현재 Final Test에서는 주문 금액의 변화를 충분히 설명하지 못했다.

Train에서 Random Forest의 오차가 더 낮았지만 Final Test에서는 Baseline보다 나빴다. 따라서 Train 성능만으로 새 주문에 대한 성능을 판단할 수 없다.

### 업무·분석적 의미

평균적으로 약 44만~46만 원의 오차가 발생했다. 이 오차를 실제 업무에서 감당할 수 있는지는 주문 금액의 규모와 예측값의 사용 목적을 함께 확인해야 한다. 현재 결과만으로 Random Forest가 평균값 예측보다 유용하다고 말할 수 없다.

### 현재 한계와 추가 확인 사항

RMSE가 더 큰 이유를 이 표만으로 특정할 수 없다. 다음 단계에서 실제값·예측값·잔차를 확인해 큰 오차 사례를 조사한다. 현재 feature에 주문 금액을 설명할 정보가 부족할 **가능성**은 있지만, 원인으로 단정하지 않는다.

### STEP 11. 대표 오차 사례를 진단합니다

In [82]:
import pandas as pd
from IPython.display import display

# STEP 9에서 만든 고정 모델의 Final Test 예측값을 가져온다.
predicted = final_predictions[selected_model_name]

# 실제값과 예측값을 같은 순서로 한 행씩 짝짓는다.
error_detail = pd.DataFrame({
    "actual_order_total": y_final_test.to_numpy(),
    "predicted_order_total": predicted,
})

# 잔차 = 실제값 - 예측값
# 양수: 실제보다 낮게 예측 / 음수: 실제보다 높게 예측
error_detail["residual"] = (
    error_detail["actual_order_total"]
    - error_detail["predicted_order_total"]
)
error_detail["abs_error"] = error_detail["residual"].abs()

# 절대오차가 가장 큰 사례 3개를 고른다.
top_errors = (
    error_detail.sort_values("abs_error", ascending=False)
    .head(3)
    .reset_index(drop=True)
)

top_errors.insert(0, "사례", ["사례 1", "사례 2", "사례 3"])
top_errors["예측 방향"] = top_errors["residual"].apply(
    lambda value: "실제보다 낮게 예측" if value > 0
    else "실제보다 높게 예측" if value < 0
    else "정확히 예측"
)

print("잔차 = 실제 주문 금액 - 예측 주문 금액")
display(top_errors.round(2))

잔차 = 실제 주문 금액 - 예측 주문 금액


,사례,actual_order_total,predicted_order_total,residual,abs_error,예측 방향
0,사례 1,1877000,605528.72,1271471.28,1271471.28,실제보다 낮게 예측
1,사례 2,1767000,756889.65,1010110.35,1010110.35,실제보다 낮게 예측
2,사례 3,1863000,861784.71,1001215.29,1001215.29,실제보다 낮게 예측


## STEP 11. 대표 오차 사례 진단

### 결과 관찰

잔차는 `실제 주문 금액 - 예측 주문 금액`으로 계산했다. 잔차가 양수이면 실제보다 낮게 예측한 것이다. Random Forest의 Final Test에서 절대오차가 큰 상위 3건은 다음과 같다.

| 사례 | 실제값 | 예측값 | 잔차 | 절대오차 |
| --- | ---: | ---: | ---: | ---: |
| 사례 1 | 1,877,000원 | 605,528.72원 | +1,271,471.28원 | 1,271,471.28원 |
| 사례 2 | 1,767,000원 | 756,889.65원 | +1,010,110.35원 | 1,010,110.35원 |
| 사례 3 | 1,863,000원 | 861,784.71원 | +1,001,215.29원 | 1,001,215.29원 |

세 사례 모두 실제보다 낮게 예측했고, 각 절대오차가 100만 원을 넘었다. 이처럼 큰 오차가 있어 RMSE가 MAE보다 높게 나온 결과를 이해하는 데 참고가 된다.

### 나의 해석과 판단

현재 입력 feature에는 주문 상세 수량·단가·상품 정보가 포함되지 않는다. 따라서 고액 주문을 구별할 정보가 부족했을 **가능성**이 있다. 다만 이 표만으로 오차의 원인을 확정할 수는 없다.

### 업무·분석적 의미

이 세 주문에서 예측값을 실제 주문 금액으로 사용했다면 각각 100만 원 이상 과소평가했을 것이다. 고액 주문에서 이런 오차가 반복되는지 확인하기 전에는 예측값을 금액 결정에 바로 사용하기 어렵다.

### 추가 확인 사항

상위 3건만으로 모든 고액 주문을 낮게 예측한다고 결론 내릴 수 없다. Final Test 전체에서 주문 금액대별 오차와 과소예측 비율을 추가로 확인해야 한다.

### 시사점

금액이 큰 주문 3건을 모델이 모두 **100만 원 이상 낮게 예측**했다. 주문 날짜와 고객 정보만으로는 고액 주문을 구별하기 어려울 가능성이 있다.

따라서 이 예측값을 매출 계획이나 금액 결정에 바로 사용하면 고액 주문을 과소평가할 위험이 있다. 다만 확인한 것은 오차가 가장 큰 3건이므로, **모든 고액 주문에서 같은 문제가 생기는지** 전체 결과를 더 확인해야 한다.

### STEP 12. 자동 Validation Evidence를 확인합니다

### STEP 12. 자동 Validation Evidence 확인

STEP 12는 **지금까지의 분석 절차가 규칙대로 진행됐는지 검사하는 작업**이다.  
금지된 feature가 입력에 섞이지 않았는지, Train과 Final Test가 시간 순서대로 나뉘었는지 확인한다.  
선택한 모델이 Train 교차검증에 있었고, Final Test에서는 Baseline과 고정 모델만 평가했는지도 확인한다.  
시험을 채점하기 전에 **시험 규칙을 제대로 지켰는지 검사하는 것**과 비슷하다.  
검사 결과에 `FAIL`이 있으면 원인을 고친 뒤 다시 확인해야 한다.

In [83]:
import sys
import subprocess
import pandas as pd
from IPython.display import display

# 현재 노트북에서 정상 작동한 Python으로 공식 저장소의 스크립트를 실행한다.
run_result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=project_root,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print("사용한 Python:", sys.executable)
print("종료 코드:", run_result.returncode)

if run_result.returncode != 0:
    # 성공 출력에는 내부 주문 ID가 있으므로 오류 출력만 표시한다.
    print("오류 원인:")
    print(run_result.stderr[-4000:])
else:
    # 공식 스크립트가 생성한 자동 검증 파일을 읽는다.
    validation = pd.read_csv(validation_path)
    display(validation[["check", "value", "status"]])
    print("모든 검증 PASS:", validation["status"].eq("PASS").all())

사용한 Python: c:\dev\llm-data-analysis-course\.venv\Scripts\python.exe
종료 코드: 0


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


모든 검증 PASS: True


## STEP 12. 자동 Validation Evidence 확인

`reports/ch09_regression_validation.csv`를 확인한 결과, 필수 검사 6개가 모두 PASS였다.

| check | value | status |
| --- | ---: | --- |
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

금지 feature와 모델 입력의 겹침은 0개였다. Train이 Final Test보다 시간상 앞서 있고, 선택 모델은 Train CV 결과에 존재한다. Final Test에는 Baseline과 고정한 선택 모델이 포함됐으며, R² 평가에 사용한 Test 주문은 56건이다.

**FAIL 항목은 없다.** 절차 검증은 통과했지만, Random Forest의 Final Test MAE가 Baseline보다 높았다는 성능 결과는 그대로 기록한다.

### STEP 12 시사점

자동 검증 6개가 모두 PASS여서 데이터 누수와 시간 분할 등 **분석 절차는 정해진 규칙을 지켰다**. 따라서 Random Forest가 Baseline보다 못한 결과도 숨기거나 모델을 바꾸지 않고 받아들여야 한다. 이번 결과는 **검증을 통과한 모델이 반드시 성능까지 좋은 것은 아니라는 점**을 보여준다.

### STEP 13. Evidence와 공개 범위를 구분합니다

In [84]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# STEP 12에서 실행한 공식 스크립트의 결과 폴더
report_dir = Path(r"C:\dev\llm-data-analysis-course-official\reports")

# 과제 안내문에 따른 파일 구분
file_groups = {
    "검증 Evidence": [
        "ch09_regression_split_summary.csv",
        "ch09_regression_feature_audit.csv",
        "ch09_regression_cv_summary.csv",
        "ch09_regression_model_comparison.csv",
        "ch09_regression_validation.csv",
        "ch09_regression_checklist.csv",
    ],
    "공개 가능한 보고서·그래프": [
        "ch09_regression_report.md",
        "figures/ch09_actual_vs_predicted.png",
        "figures/ch09_residual_histogram.png",
    ],
    "Internal 전용": [
        "ch09_regression_model_data_internal.csv",
        "ch09_regression_predictions_internal.csv",
    ],
}

rows = []

for group, file_names in file_groups.items():
    for file_name in file_names:
        path = report_dir / file_name
        id_columns = []

        # CSV는 내용 대신 컬럼 이름만 확인한다.
        if path.is_file() and path.suffix == ".csv":
            columns = pd.read_csv(path, nrows=0).columns
            id_columns = [
                column for column in columns
                if column.lower().endswith("_id")
            ]

        rows.append({
            "구분": group,
            "파일": file_name,
            "존재": path.is_file(),
            "식별자 컬럼": ", ".join(id_columns) if id_columns else "없음",
        })

evidence_inventory = pd.DataFrame(rows)
display(evidence_inventory)

print("안내문에 나온 파일이 모두 존재:",
      evidence_inventory["존재"].all())

,구분,파일,존재,식별자 컬럼
0,검증 Evidence,ch09_regression_split_summary.csv,True,없음
1,검증 Evidence,ch09_regression_feature_audit.csv,True,없음
2,검증 Evidence,ch09_regression_cv_summary.csv,True,없음
3,검증 Evidence,ch09_regression_model_comparison.csv,True,없음
4,검증 Evidence,ch09_regression_validation.csv,True,없음
5,검증 Evidence,ch09_regression_checklist.csv,True,없음
6,공개 가능한 보고서·그래프,ch09_regression_report.md,True,없음
7,공개 가능한 보고서·그래프,figures/ch09_actual_vs_predicted.png,True,없음
8,공개 가능한 보고서·그래프,figures/ch09_residual_histogram.png,True,없음
9,Internal 전용,ch09_regression_model_data_internal.csv,True,"order_id, customer_id"


안내문에 나온 파일이 모두 존재: True


## STEP 13. Evidence와 공개 범위 구분

### 확인 결과

공식 분석 스크립트가 만든 결과 파일 11개가 모두 존재했다.

- **검증 Evidence 6개:** 시간 분할, Feature Audit, Train CV, 모델 비교, 자동 Validation, 체크리스트
- **공개 가능한 보고서·그래프 3개:** 요약 보고서, 실제값·예측값 그래프, 잔차 그래프
- **Internal 전용 2개:** 모델링 데이터와 주문별 예측 결과

Internal 모델링 데이터에는 `order_id`, `customer_id`가 있고, Internal 예측 결과에는 `order_id`가 있다. 따라서 이 두 파일은 공개용 제출물에 포함하지 않는다.

### 나의 판단

분석 근거가 되는 검증 결과와 요약 그래프는 확인할 수 있도록 두고, 주문·고객 식별자가 포함된 상세 파일은 내부 진단용으로 구분한다.

이번 코드는 **파일 존재 여부와 CSV 컬럼 이름**을 확인했다. 최종 제출 전에는 공개용 노트북과 보고서 화면에도 불필요한 식별 정보가 표시되지 않는지 확인한다.

### STEP 14. 전체 파이프라인을 처음부터 다시 실행합니다

In [85]:
import sys
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path(r"C:\dev\llm-data-analysis-course-official")
report_dir = project_root / "reports"

# 1. 입력 준비 → 회귀 분석 순서로 처음부터 다시 실행한다.
scripts = [
    "prepare_ch09_data.py",
    "run_regression_analysis.py",
]

for script_name in scripts:
    script_path = project_root / "scripts" / script_name

    run_result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=project_root,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    if run_result.returncode != 0:
        raise RuntimeError(
            f"{script_name} 실행 실패:\n{run_result.stderr[-3000:]}"
        )

    print(f"{script_name}: 성공")

# 2. 다시 생성된 공식 결과 파일을 읽는다.
script_cv = pd.read_csv(report_dir / "ch09_regression_cv_summary.csv")
script_comparison = pd.read_csv(
    report_dir / "ch09_regression_model_comparison.csv"
)
script_validation = pd.read_csv(
    report_dir / "ch09_regression_validation.csv"
)

# 3. 스크립트도 Train CV MAE로 같은 모델을 선택했는지 확인한다.
script_candidates = script_cv.loc[
    script_cv["model"] != "Baseline Mean"
].sort_values(["cv_MAE_mean", "model"])

script_selected_model = script_candidates.iloc[0]["model"]
model_matches = script_selected_model == selected_model_name

print("노트북 Selected Model:", selected_model_name)
print("스크립트 Selected Model:", script_selected_model)
print("Selected Model 일치:", model_matches)

# 4. Final Test 지표를 모델별로 노트북 결과와 비교한다.
notebook_results = final_comparison.set_index("모델")
script_results = script_comparison.set_index("model")

metric_pairs = [
    ("Test MAE", "test_MAE", 0.01),
    ("Test RMSE", "test_RMSE", 0.01),
    ("Test R²", "test_R2", 0.0001),
]

comparison_rows = []

for model_name in ["Baseline Mean", selected_model_name]:
    for notebook_column, script_column, tolerance in metric_pairs:
        notebook_value = float(notebook_results.loc[model_name, notebook_column])
        script_value = float(script_results.loc[model_name, script_column])

        comparison_rows.append({
            "모델": model_name,
            "지표": notebook_column,
            "노트북": notebook_value,
            "스크립트": script_value,
            "일치": abs(notebook_value - script_value) <= tolerance,
        })

reproduction_check = pd.DataFrame(comparison_rows)
display(reproduction_check.round(4))

print("Final Test 지표 모두 일치:", reproduction_check["일치"].all())
print("자동 Validation 모두 PASS:",
      script_validation["status"].eq("PASS").all())

prepare_ch09_data.py: 성공
run_regression_analysis.py: 성공
노트북 Selected Model: Random Forest
스크립트 Selected Model: Random Forest
Selected Model 일치: True


,모델,지표,노트북,스크립트,일치
0,Baseline Mean,Test MAE,443449.6487,443449.6487,True
1,Baseline Mean,Test RMSE,522189.6996,522189.6996,True
2,Baseline Mean,Test R²,-0.0011,-0.0011,True
3,Random Forest,Test MAE,458819.9172,458819.9172,True
4,Random Forest,Test RMSE,548190.4467,548190.4467,True
5,Random Forest,Test R²,-0.1033,-0.1033,True


Final Test 지표 모두 일치: True
자동 Validation 모두 PASS: True


## STEP 14. 전체 재실행과 결과 일치 확인

- 입력 준비 스크립트 실행: **성공**
- 회귀 분석 스크립트 실행: **성공**
- 노트북 Run All: **오류 없이 완료**
- 노트북과 스크립트의 Selected Model: **Random Forest로 일치**
- Baseline과 Random Forest의 Final Test MAE·RMSE·R²: **6개 값 모두 일치**
- 자동 Validation: **모두 PASS**

처음부터 다시 실행해도 같은 모델이 선택되고 같은 Final Test 지표가 나왔다. 따라서 이번 결과는 이전 셀의 임시 메모리 상태에 의존하지 않고 재현됐다.

재실행 결과에서도 Random Forest의 Test MAE는 **458,819.92원**으로 Baseline의 **443,449.65원**보다 높았다. 성능이 낮은 결과도 바꾸지 않고 그대로 기록한다.

### STEP 15. 모델 사용 가능성을 판단하고 제출합니다

## STEP 15. 최종 사용 판단

### 최종 판단

**현재 데이터로는 사용 보류**

### 판단 근거

1. Train 교차검증에서 Random Forest의 MAE 평균은 **437,352.80원**으로 Baseline의 **423,004.81원**보다 높았다. Random Forest의 회차별 MAE 표준편차는 26,318.79원으로 상대적으로 작았지만, 평균 오차의 개선으로 이어지지는 않았다.
2. Final Test에서도 Random Forest의 MAE는 **458,819.92원**으로 Baseline보다 **3.47% 높았다**. RMSE는 **548,190.45원**, R²는 **-0.1033**이었다.
3. 오차가 가장 큰 세 주문은 모두 실제보다 **100만 원 이상 낮게 예측**했다. 고액 주문에서 이런 오류가 반복되는지는 추가 확인이 필요하다.
4. 금지 feature 겹침, 시간 분할 등 자동 Validation은 모두 PASS했고 전체 재실행 결과도 일치했다. 따라서 낮은 성능을 평가 절차의 실패로 단정하지 않고 결과 그대로 기록한다.

### 업무적 위험

예측값을 매출 계획이나 금액 결정에 사용하면 고액 주문을 과소평가할 수 있다. 실제 업무에서 허용 가능한 오차 기준도 아직 정해지지 않았으므로 현재 예측값의 사용을 보류한다.

### 현재 데이터의 한계

모델 입력은 주문 시점과 고객 특성 6개로 제한된다. 주문 상세 정보는 예측 시점에 알 수 없다는 가정에 따라 제외했으며, `payment_method`가 실제로 언제 확정되는지도 확인이 필요하다. 또한 Train 교차검증의 한 회차에서는 같은 날짜가 학습과 검증에 걸쳐 있었다.

### 다음 개선 우선순위

1. 실제 예측 시점을 확인하고, 그 시점에 이용 가능한 feature를 다시 정의한다. Target 계산 재료와 사후 정보는 계속 제외한다.
2. 주문 금액대별 오차와 과소예측 비율을 분석하고, 업무에서 허용할 오차 기준을 정한다.
3. 같은 날짜가 교차검증의 학습·검증에 겹치지 않도록 분할을 개선하고, 새로운 미래 데이터로 다시 평가한다.

In [86]:
import pandas as pd

# 가상의 새 주문 정보
order_date = pd.Timestamp("2026-10-15")

new_order = pd.DataFrame([{
    "order_month": order_date.month,
    "order_dayofweek": order_date.dayofweek,
    "age": 30,
    "payment_method": "card",
    "gender": "F",
    "city": "서울",
}])

prediction = final_models[selected_model_name].predict(new_order)[0]
print(f"예상 주문 금액: {prediction:,.0f}원")

예상 주문 금액: 875,558원
